# Attribute boosting v2

Reuses the first run's name embeddings and tests normalized/fuzzy
attributes, per-category CatBoost, and rich `name + attributes` Qwen
embeddings. All cells run sequentially and persist outputs.

In [ ]:
import hashlib, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath

INPUT = Path('/kaggle/input'); WORK = Path('/kaggle/working')
PROJECT = WORK / 'product_matching'; OUTPUT = WORK / 'attribute_boosting_v2'
EXPECTED_HASH = '0158ff463f8fbcdd3be2ca091622b9cb9c146d98eeccf7fd3023204b1d9cebee'

def exactly_one(pattern):
    values = list(INPUT.glob(pattern))
    if len(values) != 1: raise RuntimeError(f'Expected one {pattern}, found {values}')
    return values[0]

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024*1024), b''): digest.update(chunk)
    return digest.hexdigest()

items = exactly_one('**/items_human.parquet'); matches = exactly_one('**/matches.parquet')
embeddings = exactly_one('**/embedding_boosting/item_embeddings.f16.npy')
previous = embeddings.parent
bundles = list(INPUT.glob('**/product_matching_training_code.zip')) + [p for p in INPUT.glob('**/product_matching_training_code') if p.is_dir()]
if len(bundles) != 1: raise RuntimeError(f'Expected one code bundle, found {bundles}')
bundle = bundles[0]; PROJECT.mkdir(parents=True, exist_ok=True)
if bundle.is_file():
    if sha256(bundle) != EXPECTED_HASH: raise RuntimeError('Code bundle hash mismatch')
    with zipfile.ZipFile(bundle) as archive:
        for member in archive.namelist():
            path = PurePosixPath(member)
            if path.is_absolute() or '..' in path.parts: raise RuntimeError(member)
        archive.extractall(PROJECT)
else: shutil.copytree(bundle, PROJECT, dirs_exist_ok=True)
print(items, matches, previous, sep='\n')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(PROJECT/'requirements-embedding-boosting.txt')], check=True)

In [ ]:
command = [sys.executable, '-u', str(PROJECT/'src/attribute_boosting_v2.py'),
           '--items', str(items), '--matches', str(matches),
           '--config', str(PROJECT/'configs/attribute_boosting_v2.json'),
           '--previous-output', str(previous), '--output-dir', str(OUTPUT)]
print('$', ' '.join(command), flush=True)
with (WORK/'attribute_boosting_v2_console.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
    code = process.wait()
if code: raise RuntimeError(f'Experiment failed: {{code}}')
print((OUTPUT/'experiment_comparison.csv').read_text(encoding='utf-8'))